# CVRP Solver — qCourier Yale Hackathon 2026

Classical implementation of our proximity-clustering algorithm for the Capacitated Vehicle Routing Problem.

## Algorithm Overview

1. **Encode locations** — assign index `n` to each location. The depot is `alpha_0`, customers are `alpha_1, ..., alpha_H`.
2. **Build distance matrix** — compute the Euclidean distance between every pair of locations.
3. **Minimum vehicles** — compute `G = min(ceil(H/C), N)`: the fewest vehicles needed to serve all H customers within capacity C, capped at the fleet size N.
4. **Cluster houses** — sort customers by polar angle from the depot (sweep), try all rotations, keep the grouping with lowest total route distance. Each group capped at C customers.
5. **Assign vehicles** — each vehicle is assigned one group and makes one round trip.

In [1]:
import math
import os

## Step 1 — Encode Locations

In [2]:
def encode_locations(depot, customers):
    locations = {0: tuple(depot)}
    for i, c in enumerate(customers, start=1):
        locations[i] = tuple(c)
    return locations

## Step 2 — Build Distance Matrix

In [3]:
def euclidean(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def build_distance_matrix(locations):
    dist_matrix = {}
    ids = list(locations.keys())
    for i in ids:
        for j in ids:
            if i != j:
                dist_matrix[(i, j)] = euclidean(locations[i], locations[j])
    return dist_matrix

## Step 3 — Minimum Number of Vehicles (Groups)

`G = min(ceil(H / C), N)` where H is the number of customers, C is vehicle capacity, and N is the fleet size.

This gives the fewest vehicles needed to cover all customers within capacity, capped at the available fleet.

In [4]:
def compute_num_groups(H, C, N):
    return min(math.ceil(H / C), N)

## Step 4 — Cluster Houses by Sweep Algorithm

Sort customers by their polar angle from the depot, then assign consecutive customers (in angular order) to the same group. This ensures vehicles serve customers in the same directional sector, minimizing backtracking.

We try all possible starting cuts of the sweep circle and keep the one with the lowest total estimated route distance.

In [5]:
def _route_distance_estimate(group, dist_matrix):
    unvisited = list(group)
    current = 0
    total = 0.0
    while unvisited:
        nearest = min(unvisited, key=lambda c: dist_matrix[(current, c)])
        total += dist_matrix[(current, nearest)]
        current = nearest
        unvisited.remove(nearest)
    total += dist_matrix[(current, 0)]
    return total

def cluster_houses(locations, dist_matrix, G, C):
    customer_ids = [n for n in locations if n != 0]
    H = len(customer_ids)
    depot_x, depot_y = locations[0]

    sorted_customers = sorted(
        customer_ids,
        key=lambda n: math.atan2(locations[n][1] - depot_y, locations[n][0] - depot_x)
    )

    chunk_size = min(math.ceil(H / G), C)
    best_groups = None
    best_dist = float('inf')

    for start in range(H):
        rotated = sorted_customers[start:] + sorted_customers[:start]
        groups = [rotated[i:i + chunk_size] for i in range(0, H, chunk_size)]
        while len(groups) > G:
            groups[-2] = groups[-2] + groups[-1]
            groups.pop()
        groups = [g for g in groups if g]
        total = sum(_route_distance_estimate(g, dist_matrix) for g in groups)
        if total < best_dist:
            best_dist = total
            best_groups = [list(g) for g in groups]

    return best_groups

## Step 5 — Assign Vehicles to Groups

In [6]:
def assign_vehicles(groups, N):
    assignment = {v: [] for v in range(1, N + 1)}
    for i, group in enumerate(groups):
        assignment[(i % N) + 1].append(group)
    return assignment

## Route Construction

Greedy nearest-neighbor ordering within each group: depot → nearest unvisited → ... → depot.

In [7]:
from itertools import permutations

def build_route(group, dist_matrix):
    """Try all orderings of the group and return the shortest round-trip."""
    best_route = None
    best_dist = float('inf')
    for perm in permutations(group):
        route = [0] + list(perm) + [0]
        dist = sum(dist_matrix[(route[i], route[i+1])] for i in range(len(route)-1))
        if dist < best_dist:
            best_dist = dist
            best_route = route
    return best_route

def build_all_routes(assignment, dist_matrix):
    routes = []
    for v in sorted(assignment):
        for group in assignment[v]:
            routes.append((v, build_route(group, dist_matrix)))
    return routes

def total_distance(routes, dist_matrix):
    return sum(
        dist_matrix[(route[i], route[i + 1])]
        for _, route in routes
        for i in range(len(route) - 1)
    )

## Full Solver

In [8]:
def solve_cvrp(depot, customers, N, C, verbose=True):
    locations = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locations)
    H = len(customers)
    G = compute_num_groups(H, C, N)

    if verbose:
        print(f"G = min(ceil({H}/{C}), {N}) = {G} vehicle(s)")

    groups = cluster_houses(locations, dist_matrix, G, C)

    if verbose:
        print("Groups:")
        for i, g in enumerate(groups):
            print(f"  G_{i+1}: customers {g}")

    assignment = assign_vehicles(groups, N)

    routes = build_all_routes(assignment, dist_matrix)
    dist = total_distance(routes, dist_matrix)

    if verbose:
        print("Routes:")
        for i, (v, route) in enumerate(routes):
            print(f"  r{i+1} (vehicle {v}): {' -> '.join(str(n) for n in route)}")
        print(f"Total distance: {dist:.4f}")

    return routes, dist

## Run All CVRP Instances

In [9]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                            "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

results = {}
for inst_num, params in instances.items():
    print(f"{'='*50}")
    print(f"Instance {inst_num}  |  N={params['N']}, C={params['C']}")
    print(f"{'='*50}")
    routes, dist = solve_cvrp(depot, params["customers"], params["N"], params["C"])
    results[inst_num] = {"routes": routes, "total_distance": dist}
    print()

Instance 1  |  N=2, C=5
G = min(ceil(3/5), 2) = 1 vehicle(s)
Groups:
  G_1: customers [3, 2, 1]
Routes:
  r1 (vehicle 1): 0 -> 3 -> 2 -> 1 -> 0
Total distance: 21.7445

Instance 2  |  N=2, C=2
G = min(ceil(3/2), 2) = 2 vehicle(s)
Groups:
  G_1: customers [2, 1]
  G_2: customers [3]
Routes:
  r1 (vehicle 1): 0 -> 2 -> 1 -> 0
  r2 (vehicle 2): 0 -> 3 -> 0
Total distance: 26.1817

Instance 3  |  N=3, C=2
G = min(ceil(6/2), 3) = 3 vehicle(s)
Groups:
  G_1: customers [6, 4]
  G_2: customers [3, 5]
  G_3: customers [2, 1]
Routes:
  r1 (vehicle 1): 0 -> 6 -> 4 -> 0
  r2 (vehicle 2): 0 -> 3 -> 5 -> 0
  r3 (vehicle 3): 0 -> 2 -> 1 -> 0
Total distance: 50.6965

Instance 4  |  N=4, C=3
G = min(ceil(12/3), 4) = 4 vehicle(s)
Groups:
  G_1: customers [3, 5, 4]
  G_2: customers [9, 10, 6]
  G_3: customers [12, 11, 2]
  G_4: customers [7, 1, 8]
Routes:
  r1 (vehicle 1): 0 -> 5 -> 3 -> 4 -> 0
  r2 (vehicle 2): 0 -> 9 -> 10 -> 6 -> 0
  r3 (vehicle 3): 0 -> 12 -> 11 -> 2 -> 0
  r4 (vehicle 4): 0 -> 7 -> 

## Write Solution Files

Outputs `solutions/Instance{N}.txt` in the required submission format.

In [10]:
os.makedirs("solutions", exist_ok=True)

for inst_num, res in results.items():
    path = f"solutions/Instance{inst_num}.txt"
    with open(path, "w") as f:
        for i, (_, route) in enumerate(res["routes"], start=1):
            f.write("r" + str(i) + ": " + ", ".join(str(n) for n in route) + "\n")
    print(f"Written {path}")

Written solutions/Instance1.txt
Written solutions/Instance2.txt
Written solutions/Instance3.txt
Written solutions/Instance4.txt
